In [2]:
from IPython.display import Video
Video("/home/namnh1/zalo-challenge/video_test/media_20241001_081533_8423303510524980008.mp4")


In [3]:
import cv2
import numpy as np
import os
import warnings

from src.generate_patches import CropImage
from src.model_test import AntiSpoofPredict
from src.utility import parse_model_name

warnings.filterwarnings("ignore")


def test(video_path, model_dir, output_video_path):
    model_test = AntiSpoofPredict()
    image_cropper = CropImage()
    cap = cv2.VideoCapture(video_path)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video_path, fourcc, 20.0, 
                          (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Get bounding box
        image_bbox, bbox_org = model_test.get_bbox(frame)
        prediction = np.zeros((1, 2))

        for model_name in os.listdir(model_dir):
            h_input, w_input, model_type, scale = parse_model_name(model_name)
            param = {
                "org_img": frame,
                "bbox": image_bbox,
                "scale": scale,
                "out_w": w_input,
                "out_h": h_input,
                "crop": True,
            }
            if scale is None:
                param["crop"] = False
            img = image_cropper.crop(**param)
            prediction += model_test.predict(img, os.path.join(model_dir, model_name))

        # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        label = np.argmax(prediction)
        value = prediction[0][label] / 3
        if label == 1 and value > 0.8:
            spoof_confidence = value
            text = f"Label: Real, Confidence: {spoof_confidence:.2f}"
            color = (0, 255, 0)
        else:
            spoof_confidence = 1 - value
            text = f"Label: 'Spoof', Confidence: {spoof_confidence:.2f}"
            color = (0, 0, 255)

        # Visualize: draw bbox and prediction result on the frame
        x1, y1, x2, y2 = map(int, bbox_org)  # assuming get_bbox returns [x1, y1, x2, y2]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
        print(text)
        # Save the frame to the output video
        out.write(frame)

    cap.release()
    out.release()
    print(f"Processed video saved at {output_video_path}")

def processing_folder(folder):
    for i, video_name in enumerate(os.listdir(folder)):
        print(video_name)
        test(os.path.join(folder, video_name), "resources/test_5", f"./results/output_video_{i}.mp4")

def processing_image(video_path):
    test(video_path, "resources/test_5", "output_video.mp4")
    

if __name__ == "__main__":
    processing_image("/home/namnh1/zalo-challenge/datasets/video-tiktok/IMG_3305.mov")


OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


Label: 'Spoof', Confidence: 0.37
Label: 'Spoof', Confidence: 0.34
Label: 'Spoof', Confidence: 0.18
Label: 'Spoof', Confidence: 0.35
Label: 'Spoof', Confidence: 0.37
Label: 'Spoof', Confidence: 0.36
Label: 'Spoof', Confidence: 0.36
Label: 'Spoof', Confidence: 0.30
Label: 'Spoof', Confidence: 0.30
Label: 'Spoof', Confidence: 0.33
Label: 'Spoof', Confidence: 0.31
Label: 'Spoof', Confidence: 0.29
Label: 'Spoof', Confidence: 0.25
Label: 'Spoof', Confidence: 0.32
Label: 'Spoof', Confidence: 0.25
Label: 'Spoof', Confidence: 0.38
Label: 'Spoof', Confidence: 0.34
Label: 'Spoof', Confidence: 0.36
Label: 'Spoof', Confidence: 0.28
Label: 'Spoof', Confidence: 0.32
Label: 'Spoof', Confidence: 0.31
Label: 'Spoof', Confidence: 0.27
Label: 'Spoof', Confidence: 0.29
Label: 'Spoof', Confidence: 0.25
Label: 'Spoof', Confidence: 0.32
Label: 'Spoof', Confidence: 0.33
Label: 'Spoof', Confidence: 0.09
Label: 'Spoof', Confidence: 0.32
Label: 'Spoof', Confidence: 0.28
Label: 'Spoof', Confidence: 0.29
Label: 'Sp

In [3]:
import cv2
import os
import numpy as np
import pandas as pd

from src.generate_patches import CropImage
from src.model_test import AntiSpoofPredict
from src.utility import parse_model_name



def test(image, model_dir):
    model_test = AntiSpoofPredict()
    image_cropper = CropImage()

    image_bbox, bbox_org = model_test.get_bbox(image)
    prediction = np.zeros((1, 2))

    for model_name in os.listdir(model_dir):
        h_input, w_input, model_type, scale = parse_model_name(model_name)
        param = {
            "org_img": image,
            "bbox": image_bbox,
            "scale": scale,
            "out_w": w_input,
            "out_h": h_input,
            "crop": True,
        }
        if scale is None:
            param["crop"] = False
        img = image_cropper.crop(**param)
        prediction += model_test.predict(img, os.path.join(model_dir, model_name))

    label = np.argmax(prediction)
    value = prediction[0][label] / len(os.listdir(model_dir))
    if label == 1 and value > 0.8:
        spoof_confidence = value
        text = f"Label: Real, Confidence: {spoof_confidence:.2f}"
        color = (0, 255, 0)
    else:
        spoof_confidence = 1 - value
        text = f"Label: 'Spoof', Confidence: {spoof_confidence:.2f}"
        color = (0, 0, 255)
    print(text)
    # Visualize: draw bbox and prediction result on the frame
    x1, y1, x2, y2 = map(int, bbox_org)  # assuming get_bbox returns [x1, y1, x2, y2]
    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
    cv2.putText(image, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv2.imwrite(f"/home/namnh1/zalo-challenge/datasets/results/res_2.jpg", image)
    

In [19]:
image = cv2.imread("/home/namnh1/zalo-challenge/image_spoof_1360a76.jpg")
test(image, "/home/namnh1/zalo-challenge/resources/test_6")

Label: Real, Confidence: 0.94
